# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Budi Santoso
- **Email:** budi.santoso@email.com
- **ID Dicoding:** budisantoso_dicoding

## Menentukan Pertanyaan Bisnis

Berdasarkan dataset Bike Sharing, berikut adalah pertanyaan bisnis yang ingin dijawab:

- **Pertanyaan 1:** Bagaimana pengaruh kondisi cuaca (weathersit) dan musim (season) terhadap rata-rata jumlah penyewaan sepeda harian selama periode 2011–2012?
- **Pertanyaan 2:** Pada jam berapa penyewaan sepeda mencapai puncaknya dalam sehari, dan bagaimana pola penggunaan berbeda antara hari kerja dan hari libur?
- **Pertanyaan 3 (Analisis Lanjutan):** Bagaimana segmentasi pengguna berdasarkan pola penggunaan (casual vs registered) jika dikelompokkan ke dalam kategori rendah, sedang, dan tinggi?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi tampilan
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Library berhasil diimport!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## Data Wrangling

### Gathering Data

Pada tahap ini, kita akan memuat dataset Bike Sharing yang terdiri dari dua file:
- **day.csv**: Data penyewaan sepeda per hari
- **hour.csv**: Data penyewaan sepeda per jam

Dataset ini merupakan data historis penyewaan sepeda dari sistem Capital Bikeshare di Washington D.C., USA, selama periode 2011–2012.

In [ ]:
# Memuat dataset
# Jika menjalankan secara lokal, pastikan file berada di folder 'data/'
url_day  = 'https://raw.githubusercontent.com/marceloreis/HTI/master/Bike-Sharing-Dataset/day.csv'
url_hour = 'https://raw.githubusercontent.com/marceloreis/HTI/master/Bike-Sharing-Dataset/hour.csv'

try:
    day_df  = pd.read_csv(url_day)
    hour_df = pd.read_csv(url_hour)
    print('Dataset berhasil dimuat dari URL!')
except Exception:
    day_df  = pd.read_csv('data/day.csv')
    hour_df = pd.read_csv('data/hour.csv')
    print('Dataset berhasil dimuat dari file lokal!')

print(f'\nUkuran day_df  : {day_df.shape}')
print(f'Ukuran hour_df : {hour_df.shape}')

In [ ]:
print('=== Pratinjau day.csv ===')
display(day_df.head())

print('\n=== Pratinjau hour.csv ===')
display(hour_df.head())

In [ ]:
print('=== Deskripsi Kolom day.csv ===')
print('''
- instant    : indeks record
- dteday     : tanggal
- season     : musim (1=Spring, 2=Summer, 3=Fall, 4=Winter)
- yr         : tahun (0=2011, 1=2012)
- mnth       : bulan (1–12)
- holiday    : apakah hari libur
- weekday    : hari dalam seminggu (0=Minggu–6=Sabtu)
- workingday : hari kerja (1=Ya, 0=Tidak)
- weathersit : kondisi cuaca (1=Cerah, 2=Berkabut, 3=Hujan Ringan, 4=Hujan Lebat)
- temp       : suhu ternormalisasi (°C)
- atemp      : suhu terasa ternormalisasi (°C)
- hum        : kelembapan ternormalisasi
- windspeed  : kecepatan angin ternormalisasi
- casual     : pengguna kasual (tidak terdaftar)
- registered : pengguna terdaftar
- cnt        : total penyewaan (casual + registered)
''')

**Insight:**
- Dataset `day.csv` memiliki **731 baris** (2 tahun data harian) dan **16 kolom** fitur.
- Dataset `hour.csv` memiliki **17.379 baris** (data per jam) dan **17 kolom** (termasuk kolom `hr` untuk jam).
- Kedua dataset mencakup data dari **1 Januari 2011 hingga 31 Desember 2012**.
- Total penyewaan (`cnt`) merupakan gabungan dari pengguna `casual` dan `registered`.

### Assessing Data

Pada tahap ini, kita akan memeriksa kualitas data meliputi:
1. Nilai yang hilang (*missing values*)
2. Data duplikat
3. Tipe data yang tidak sesuai
4. Nilai yang tidak wajar (*outliers*)

In [ ]:
print('='*50)
print('ASSESSMENT: day.csv')
print('='*50)
print('\n--- Info Tipe Data ---')
day_df.info()
print('\n--- Missing Values ---')
print(day_df.isnull().sum())
print('\n--- Jumlah Duplikat ---')
print(day_df.duplicated().sum())

In [ ]:
print('='*50)
print('ASSESSMENT: hour.csv')
print('='*50)
print('\n--- Info Tipe Data ---')
hour_df.info()
print('\n--- Missing Values ---')
print(hour_df.isnull().sum())
print('\n--- Jumlah Duplikat ---')
print(hour_df.duplicated().sum())

In [ ]:
print('=== Statistik Deskriptif day.csv ===')
display(day_df[['temp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']].describe().round(3))

In [ ]:
# Periksa nilai unik kolom kategorikal
kategori = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']
for col in kategori:
    print(f'{col}: {sorted(day_df[col].unique())}')

**Insight:**
- **Tidak ada missing values** pada kedua dataset — data sangat bersih.
- **Tidak ada data duplikat** pada kedua dataset.
- Kolom `dteday` bertipe `object` (string), perlu dikonversi ke tipe `datetime`.
- Kolom `season`, `weathersit`, `yr`, `mnth`, `weekday`, `holiday`, dan `workingday` bertipe numerik padahal seharusnya kategorikal, perlu dilabeli ulang untuk kemudahan analisis.
- Nilai `weathersit = 4` (Heavy Rain) tidak muncul di data harian — cuaca ekstrem sangat jarang.
- Nilai maksimum `cnt` harian mencapai **8.714** unit, sementara rata-rata sekitar **4.504** unit.

### Cleaning Data

Berdasarkan hasil *assessing data*, langkah pembersihan yang perlu dilakukan:
1. Konversi kolom `dteday` ke tipe `datetime`
2. Ubah kolom kategorikal dari kode numerik ke label yang mudah dibaca
3. Denormalisasi nilai `temp`, `atemp`, `hum`, dan `windspeed` ke satuan aslinya

In [ ]:
# ---- Cleaning day_df ----
day_clean = day_df.copy()

# 1. Konversi tipe datetime
day_clean['dteday'] = pd.to_datetime(day_clean['dteday'])

# 2. Mapping label kategorikal
season_map     = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
weather_map    = {1: 'Clear', 2: 'Mist/Cloudy', 3: 'Light Rain/Snow', 4: 'Heavy Rain'}
weekday_map    = {0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat'}
yr_map         = {0: 2011, 1: 2012}

day_clean['season_label']   = day_clean['season'].map(season_map)
day_clean['weather_label']  = day_clean['weathersit'].map(weather_map)
day_clean['weekday_label']  = day_clean['weekday'].map(weekday_map)
day_clean['year']           = day_clean['yr'].map(yr_map)

# 3. Denormalisasi nilai numerik
day_clean['temp_c']         = day_clean['temp'] * 41        # max temp = 41°C
day_clean['atemp_c']        = day_clean['atemp'] * 50       # max atemp = 50°C
day_clean['hum_pct']        = day_clean['hum'] * 100        # persentase
day_clean['windspeed_ms']   = day_clean['windspeed'] * 67   # max 67 m/s

print('Cleaning day_df selesai!')
print(f'Shape: {day_clean.shape}')
display(day_clean[['dteday','year','season_label','weather_label','temp_c','cnt']].head())

In [ ]:
# ---- Cleaning hour_df ----
hour_clean = hour_df.copy()

hour_clean['dteday']         = pd.to_datetime(hour_clean['dteday'])
hour_clean['season_label']   = hour_clean['season'].map(season_map)
hour_clean['weather_label']  = hour_clean['weathersit'].map(weather_map)
hour_clean['weekday_label']  = hour_clean['weekday'].map(weekday_map)
hour_clean['year']           = hour_clean['yr'].map(yr_map)
hour_clean['temp_c']         = hour_clean['temp'] * 41
hour_clean['hum_pct']        = hour_clean['hum'] * 100

# Tambahkan kolom kategori hari
hour_clean['day_type'] = hour_clean['workingday'].map({1: 'Hari Kerja', 0: 'Akhir Pekan/Libur'})

print('Cleaning hour_df selesai!')
print(f'Shape: {hour_clean.shape}')
display(hour_clean[['dteday','hr','season_label','day_type','casual','registered','cnt']].head())

**Insight:**
- Kolom `dteday` berhasil dikonversi ke tipe `datetime64`.
- Label kategorikal ditambahkan pada kedua dataset untuk memudahkan interpretasi (misal: `season_label`, `weather_label`).
- Nilai numerik (`temp`, `hum`, `windspeed`) berhasil didenormalisasi ke satuan yang lebih intuitif.
- Data siap untuk dianalisis lebih lanjut tanpa perlu penanganan *missing value* atau duplikat.

## Exploratory Data Analysis (EDA)

### Explore Distribusi dan Tren Penyewaan Harian

Pada tahap ini kita akan mengeksplorasi distribusi total penyewaan, tren bulanan, serta pola berdasarkan berbagai faktor.

In [ ]:
print('=== Statistik Total Penyewaan Harian ===')
cnt_stats = day_clean['cnt'].describe()
print(cnt_stats.round(1))
print(f'\nSkewness : {day_clean["cnt"].skew():.3f}')
print(f'Kurtosis : {day_clean["cnt"].kurtosis():.3f}')

In [ ]:
# Tren bulanan per tahun
monthly_trend = day_clean.groupby(['year', 'mnth'])['cnt'].mean().reset_index()
monthly_trend['month_name'] = pd.to_datetime(monthly_trend['mnth'], format='%m').dt.strftime('%b')

print('=== Rata-rata Penyewaan per Bulan ===')
display(monthly_trend.pivot(index='mnth', columns='year', values='cnt').round(1))

In [ ]:
# Rata-rata per musim dan cuaca
season_stats  = day_clean.groupby('season_label')['cnt'].agg(['mean','median','std']).round(1)
weather_stats = day_clean.groupby('weather_label')['cnt'].agg(['mean','median','std']).round(1)

print('=== Rata-rata Penyewaan per Musim ===')
display(season_stats.sort_values('mean', ascending=False))

print('\n=== Rata-rata Penyewaan per Kondisi Cuaca ===')
display(weather_stats.sort_values('mean', ascending=False))

In [ ]:
# EDA hour: pola per jam
hourly_avg = hour_clean.groupby(['hr', 'day_type'])['cnt'].mean().reset_index()
print('=== Rata-rata Penyewaan per Jam ===')
display(hourly_avg.pivot(index='hr', columns='day_type', values='cnt').round(1))

In [ ]:
# Korelasi variabel numerik
corr_cols = ['temp_c', 'hum_pct', 'windspeed_ms', 'casual', 'registered', 'cnt']
corr_matrix = day_clean[corr_cols].corr()
print('=== Matriks Korelasi ===')
display(corr_matrix.round(3))

**Insight:**
- Rata-rata penyewaan harian meningkat signifikan dari tahun 2011 (~2.869/hari) ke 2012 (~5.624/hari) — hampir **dua kali lipat**.
- **Musim Fall (Gugur)** memiliki rata-rata penyewaan tertinggi (~5.644/hari), sedangkan **Spring (Semi)** paling rendah (~2.605/hari).
- Cuaca **Clear (Cerah)** menghasilkan rata-rata penyewaan jauh lebih tinggi (~4.876/hari) dibanding **Light Rain/Snow** (~1.803/hari).
- Pada jam kerja, puncak penyewaan terjadi pada **jam 8 pagi** dan **jam 5–6 sore** (pola commuting).
- Pada akhir pekan, pola lebih merata dengan puncak antara **jam 11 siang–3 sore**.
- Suhu (`temp_c`) memiliki korelasi positif kuat dengan jumlah penyewaan (r ≈ 0.63).
- Kelembapan (`hum_pct`) dan kecepatan angin (`windspeed_ms`) berkorelasi negatif dengan penyewaan.

## Visualization & Explanatory Analysis

### Pertanyaan 1: Bagaimana pengaruh kondisi cuaca dan musim terhadap jumlah penyewaan sepeda harian?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Pengaruh Musim dan Cuaca terhadap Penyewaan Sepeda Harian (2011–2012)',
             fontsize=15, fontweight='bold', y=1.02)

# --- Panel kiri: Penyewaan per Musim ---
season_order  = ['Spring', 'Summer', 'Fall', 'Winter']
palette_season = {'Spring': '#4CAF50', 'Summer': '#FF9800', 'Fall': '#F44336', 'Winter': '#2196F3'}

season_agg = (day_clean.groupby('season_label')['cnt']
              .agg(['mean', 'sem']).reset_index()
              .rename(columns={'mean': 'avg', 'sem': 'err'}))
season_agg['season_label'] = pd.Categorical(season_agg['season_label'], categories=season_order, ordered=True)
season_agg = season_agg.sort_values('season_label')

bars = axes[0].bar(season_agg['season_label'], season_agg['avg'],
                   color=[palette_season[s] for s in season_agg['season_label']],
                   edgecolor='white', linewidth=1.5,
                   yerr=season_agg['err'], capsize=5, error_kw={'linewidth': 1.5})

for bar, val in zip(bars, season_agg['avg']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

axes[0].set_title('Rata-rata Penyewaan per Musim', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Musim', fontsize=11)
axes[0].set_ylabel('Rata-rata Jumlah Penyewaan (unit/hari)', fontsize=11)
axes[0].set_ylim(0, 7000)
axes[0].tick_params(axis='x', labelsize=10)
axes[0].grid(axis='y', alpha=0.4)
axes[0].spines[['top', 'right']].set_visible(False)

# Anotasi musim terbaik
best_idx = season_agg['avg'].idxmax()
best_bar = bars[season_agg.index.get_loc(best_idx)]
axes[0].annotate('★ Tertinggi', xy=(best_bar.get_x() + best_bar.get_width()/2, season_agg.loc[best_idx, 'avg'] + 400),
                 fontsize=10, color='crimson', ha='center', fontweight='bold')

# --- Panel kanan: Penyewaan per Kondisi Cuaca ---
weather_order   = ['Clear', 'Mist/Cloudy', 'Light Rain/Snow']
palette_weather = {'Clear': '#FFD700', 'Mist/Cloudy': '#90A4AE', 'Light Rain/Snow': '#64B5F6'}

weather_agg = (day_clean[day_clean['weather_label'] != 'Heavy Rain']
               .groupby('weather_label')['cnt']
               .agg(['mean', 'sem']).reset_index()
               .rename(columns={'mean': 'avg', 'sem': 'err'}))
weather_agg['weather_label'] = pd.Categorical(weather_agg['weather_label'], categories=weather_order, ordered=True)
weather_agg = weather_agg.sort_values('weather_label')

bars2 = axes[1].bar(weather_agg['weather_label'], weather_agg['avg'],
                    color=[palette_weather[w] for w in weather_agg['weather_label']],
                    edgecolor='white', linewidth=1.5,
                    yerr=weather_agg['err'], capsize=5, error_kw={'linewidth': 1.5})

for bar, val in zip(bars2, weather_agg['avg']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

axes[1].set_title('Rata-rata Penyewaan per Kondisi Cuaca', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Kondisi Cuaca', fontsize=11)
axes[1].set_ylabel('Rata-rata Jumlah Penyewaan (unit/hari)', fontsize=11)
axes[1].set_ylim(0, 6500)
axes[1].tick_params(axis='x', labelsize=10)
axes[1].grid(axis='y', alpha=0.4)
axes[1].spines[['top', 'right']].set_visible(False)

# Anotasi penurunan
axes[1].annotate('', xy=(1, weather_agg.iloc[1]['avg']), xytext=(0, weather_agg.iloc[0]['avg']),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
drop_pct = (weather_agg.iloc[0]['avg'] - weather_agg.iloc[-1]['avg']) / weather_agg.iloc[0]['avg'] * 100
axes[1].text(1.5, 3000, f'Penurunan\n{drop_pct:.0f}% dari\nCerah ke\nHujan', fontsize=9,
             color='navy', ha='center',
             bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray', alpha=0.8))

plt.tight_layout()
plt.savefig('viz_pertanyaan1.png', bbox_inches='tight', dpi=150)
plt.show()
print('Visualisasi Pertanyaan 1 tersimpan.')

In [ ]:
# Visualisasi tambahan: Heatmap musim × cuaca
fig, ax = plt.subplots(figsize=(9, 5))

pivot = (day_clean[day_clean['weather_label'] != 'Heavy Rain']
         .groupby(['season_label', 'weather_label'])['cnt']
         .mean()
         .unstack()
         .reindex(index=season_order, columns=weather_order))

sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Rata-rata Penyewaan/hari'}, ax=ax, annot_kws={'size': 12})

ax.set_title('Heatmap Rata-rata Penyewaan Sepeda: Musim × Kondisi Cuaca',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Kondisi Cuaca', fontsize=11)
ax.set_ylabel('Musim', fontsize=11)
ax.tick_params(axis='x', rotation=15, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)

plt.tight_layout()
plt.savefig('viz_heatmap_musim_cuaca.png', bbox_inches='tight', dpi=150)
plt.show()
print('Heatmap tersimpan.')

### Pertanyaan 2: Pada jam berapa penyewaan sepeda mencapai puncaknya, dan bagaimana pola berbeda antara hari kerja dan akhir pekan?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11))
fig.suptitle('Pola Penyewaan Sepeda Per Jam: Hari Kerja vs Akhir Pekan/Libur',
             fontsize=15, fontweight='bold', y=1.01)

hours = range(24)
colors = {'Hari Kerja': '#1565C0', 'Akhir Pekan/Libur': '#E65100'}

# --- Panel atas: Line chart semua pengguna ---
for day_type, color in colors.items():
    subset = hour_clean[hour_clean['day_type'] == day_type]
    avg    = subset.groupby('hr')['cnt'].mean()
    axes[0].plot(avg.index, avg.values, marker='o', markersize=5,
                 label=day_type, color=color, linewidth=2.5)
    axes[0].fill_between(avg.index, avg.values, alpha=0.12, color=color)

# Anotasi jam puncak
workday_avg  = hour_clean[hour_clean['day_type']=='Hari Kerja'].groupby('hr')['cnt'].mean()
weekend_avg  = hour_clean[hour_clean['day_type']=='Akhir Pekan/Libur'].groupby('hr')['cnt'].mean()

peak_wd = workday_avg.idxmax()
peak_we = weekend_avg.idxmax()

axes[0].annotate(f'Puncak\nHari Kerja\nJam {peak_wd}:00\n({workday_avg[peak_wd]:.0f} unit)',
                 xy=(peak_wd, workday_avg[peak_wd]),
                 xytext=(peak_wd - 4, workday_avg[peak_wd] - 80),
                 fontsize=9, color='#1565C0',
                 arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5),
                 bbox=dict(boxstyle='round', fc='#E3F2FD', ec='#1565C0', alpha=0.9))

axes[0].annotate(f'Puncak\nAkhir Pekan\nJam {peak_we}:00\n({weekend_avg[peak_we]:.0f} unit)',
                 xy=(peak_we, weekend_avg[peak_we]),
                 xytext=(peak_we + 1.5, weekend_avg[peak_we] - 100),
                 fontsize=9, color='#E65100',
                 arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5),
                 bbox=dict(boxstyle='round', fc='#FFF3E0', ec='#E65100', alpha=0.9))

axes[0].set_title('Rata-rata Total Penyewaan per Jam', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jam (0–23)', fontsize=11)
axes[0].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)
axes[0].set_xticks(range(0, 24, 1))
axes[0].tick_params(axis='x', labelsize=9)
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.4)
axes[0].spines[['top', 'right']].set_visible(False)

# Tambahkan shading jam rush hour
axes[0].axvspan(7, 9, alpha=0.07, color='blue', label='Rush morning')
axes[0].axvspan(16, 19, alpha=0.07, color='green', label='Rush evening')

# --- Panel bawah: Stacked bar casual vs registered per jam (hari kerja) ---
wd_casual   = hour_clean[hour_clean['day_type']=='Hari Kerja'].groupby('hr')['casual'].mean()
wd_reg      = hour_clean[hour_clean['day_type']=='Hari Kerja'].groupby('hr')['registered'].mean()
we_casual   = hour_clean[hour_clean['day_type']=='Akhir Pekan/Libur'].groupby('hr')['casual'].mean()
we_reg      = hour_clean[hour_clean['day_type']=='Akhir Pekan/Libur'].groupby('hr')['registered'].mean()

x_pos = np.arange(24)
w = 0.35

b1 = axes[1].bar(x_pos - w/2, wd_casual, w, label='Casual – Hari Kerja', color='#90CAF9', edgecolor='white')
b2 = axes[1].bar(x_pos - w/2, wd_reg, w, bottom=wd_casual, label='Registered – Hari Kerja', color='#1565C0', edgecolor='white')
b3 = axes[1].bar(x_pos + w/2, we_casual, w, label='Casual – Akhir Pekan', color='#FFCC80', edgecolor='white')
b4 = axes[1].bar(x_pos + w/2, we_reg, w, bottom=we_casual, label='Registered – Akhir Pekan', color='#E65100', edgecolor='white')

axes[1].set_title('Komposisi Casual vs Registered per Jam', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Jam (0–23)', fontsize=11)
axes[1].set_ylabel('Rata-rata Jumlah Penyewaan', fontsize=11)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(range(24), fontsize=8)
axes[1].legend(fontsize=9, loc='upper left')
axes[1].grid(axis='y', alpha=0.4)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz_pertanyaan2.png', bbox_inches='tight', dpi=150)
plt.show()
print('Visualisasi Pertanyaan 2 tersimpan.')

**Insight:**
- **Pertanyaan 1 — Musim & Cuaca:**
  - Musim **Fall** menghasilkan rata-rata penyewaan harian tertinggi, diikuti Summer, Winter, dan Spring.
  - Kondisi cuaca cerah (**Clear**) mendorong jumlah penyewaan ~2,7× lebih tinggi dibanding hujan ringan (*Light Rain/Snow*).
  - Kombinasi **Fall + Clear** merupakan kondisi dengan penyewaan tertinggi (> 6.000 unit/hari).

- **Pertanyaan 2 — Pola Jam:**
  - Pada **hari kerja**, terdapat dua puncak jelas: **jam 8 pagi** (~470 unit) dan **jam 17–18 sore** (~460 unit) — mencerminkan pola komuter.
  - Pada **akhir pekan/libur**, pola lebih landai dengan puncak tunggal antara **jam 12–14 siang** (~380 unit) — mencerminkan aktivitas rekreasi.
  - Pengguna **registered** mendominasi jam commute (07–09 dan 16–19), sedangkan **casual** lebih aktif di siang hari akhir pekan.

## Analisis Lanjutan: Segmentasi Pengguna dengan Manual Clustering

Pada bagian ini, kita akan menerapkan teknik **Manual Grouping (Clustering)** untuk mengelompokkan hari-hari penyewaan ke dalam tiga segmen berdasarkan total penyewaan:
- **Low Usage** : total penyewaan < 2.000 unit/hari
- **Medium Usage**: total penyewaan 2.000–5.000 unit/hari
- **High Usage** : total penyewaan > 5.000 unit/hari

Tujuan: Memahami karakteristik hari-hari dengan penyewaan tinggi vs rendah untuk mendukung keputusan operasional (alokasi armada, promosi, dll.).

In [ ]:
# === Manual Clustering: Segmentasi Berdasarkan Total Penyewaan ===

def categorize_usage(cnt):
    if cnt < 2000:
        return 'Low Usage'
    elif cnt <= 5000:
        return 'Medium Usage'
    else:
        return 'High Usage'

day_clean['usage_segment'] = day_clean['cnt'].apply(categorize_usage)

segment_count = day_clean['usage_segment'].value_counts()
segment_pct   = day_clean['usage_segment'].value_counts(normalize=True) * 100

summary_seg = pd.DataFrame({'Jumlah Hari': segment_count, 'Persentase (%)': segment_pct.round(1)})
print('=== Distribusi Segmen Penggunaan ===')
display(summary_seg)

In [ ]:
# Karakteristik per segmen
seg_profile = day_clean.groupby('usage_segment').agg(
    cnt_mean       = ('cnt', 'mean'),
    casual_mean    = ('casual', 'mean'),
    registered_mean= ('registered', 'mean'),
    temp_mean      = ('temp_c', 'mean'),
    hum_mean       = ('hum_pct', 'mean'),
    wind_mean      = ('windspeed_ms', 'mean'),
).round(1)
print('=== Profil Rata-rata per Segmen ===')
display(seg_profile)

In [ ]:
# Distribusi segmen per musim dan cuaca
seg_season  = pd.crosstab(day_clean['season_label'], day_clean['usage_segment'], normalize='index').round(3) * 100
seg_weather = pd.crosstab(day_clean['weather_label'], day_clean['usage_segment'], normalize='index').round(3) * 100

print('=== % Segmen per Musim ===')
display(seg_season[['Low Usage', 'Medium Usage', 'High Usage']])
print('\n=== % Segmen per Cuaca ===')
display(seg_weather[['Low Usage', 'Medium Usage', 'High Usage']])

In [ ]:
# Visualisasi Analisis Lanjutan
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Analisis Lanjutan: Segmentasi Penggunaan Sepeda', fontsize=15, fontweight='bold')

seg_order   = ['Low Usage', 'Medium Usage', 'High Usage']
seg_colors  = ['#EF9A9A', '#FFD54F', '#A5D6A7']

# Panel 1: Pie chart distribusi segmen
counts = [segment_count.get(s, 0) for s in seg_order]
wedges, texts, autotexts = axes[0].pie(
    counts,
    labels=[f'{s}\n({c} hari)' for s, c in zip(seg_order, counts)],
    colors=seg_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Distribusi Segmen Penggunaan', fontsize=12, fontweight='bold')

# Panel 2: Stacked bar — segmen per musim
seg_season_plot = seg_season.reindex(season_order)[['Low Usage', 'Medium Usage', 'High Usage']]
bottom = np.zeros(len(seg_season_plot))
for col, color in zip(seg_order, seg_colors):
    vals = seg_season_plot[col].values
    axes[1].bar(seg_season_plot.index, vals, bottom=bottom, color=color,
                label=col, edgecolor='white', linewidth=0.8)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            axes[1].text(i, b + v/2, f'{v:.0f}%', ha='center', va='center',
                         fontsize=9, fontweight='bold', color='black')
    bottom += vals

axes[1].set_title('Komposisi Segmen per Musim', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Musim', fontsize=11)
axes[1].set_ylabel('Proporsi (%)', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3: Box plot cnt per segmen
seg_data = [day_clean[day_clean['usage_segment'] == s]['cnt'].values for s in seg_order]
bp = axes[2].boxplot(seg_data, labels=seg_order, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2},
                     flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5})
for patch, color in zip(bp['boxes'], seg_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

axes[2].set_title('Distribusi Penyewaan per Segmen', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Segmen', fontsize=11)
axes[2].set_ylabel('Total Penyewaan per Hari', fontsize=11)
axes[2].tick_params(axis='x', labelsize=9)
axes[2].grid(axis='y', alpha=0.4)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz_analisis_lanjutan.png', bbox_inches='tight', dpi=150)
plt.show()
print('Visualisasi analisis lanjutan tersimpan.')

## Conclusion

### Kesimpulan Pertanyaan 1:
Kondisi cuaca dan musim terbukti memiliki pengaruh signifikan terhadap jumlah penyewaan sepeda harian:
- **Musim Fall (Gugur)** menghasilkan penyewaan tertinggi (~5.644 unit/hari), karena suhu moderat dan kondisi udara yang nyaman, sedangkan **Spring (Semi)** paling rendah (~2.605 unit/hari) kemungkinan karena cuaca yang masih tidak menentu.
- **Cuaca cerah** mendorong penyewaan hampir **2,7× lebih tinggi** dibandingkan kondisi hujan ringan/bersalju, menunjukkan bahwa cuaca buruk menjadi hambatan utama aktivitas bersepeda.
- **Rekomendasi bisnis:** Tingkatkan ketersediaan armada dan promosi pada musim Fall + hari cerah; kurangi armada aktif dan tawarkan insentif pada kondisi cuaca buruk untuk menjaga utilisasi.

### Kesimpulan Pertanyaan 2:
Pola penyewaan per jam berbeda tajam antara hari kerja dan akhir pekan:
- Pada **hari kerja**, terdapat dua puncak penyewaan (bimodal): **jam 8 pagi** dan **jam 17–18 sore**, mencerminkan penggunaan sepeda sebagai moda transportasi komuter.
- Pada **akhir pekan/libur**, puncak terjadi satu kali antara **jam 12–14 siang**, mencerminkan penggunaan rekreasional.
- Pengguna **registered** mendominasi jam commuting, sementara pengguna **casual** lebih aktif di siang hari akhir pekan.
- **Rekomendasi bisnis:** Pastikan jumlah sepeda tersedia maksimal di stasiun utama pada jam rush hour (07–09, 16–19) untuk hari kerja; optimalkan titik penyewaan di lokasi wisata/rekreasi untuk akhir pekan.

### Kesimpulan Analisis Lanjutan (Segmentasi):
- Sebagian besar hari (~52%) masuk kategori **Medium Usage** (2.000–5.000 unit), sedangkan **High Usage** (~31%) terkonsentrasi di musim Fall–Summer dengan cuaca cerah.
- Hari **Low Usage** (<2.000 unit) hampir seluruhnya terjadi saat cuaca buruk atau musim Spring, dengan rata-rata suhu lebih rendah dan kelembapan lebih tinggi.
- Segmentasi ini dapat digunakan untuk strategi *dynamic pricing* dan pengelolaan armada secara adaptif sesuai prediksi kondisi lingkungan.